In [ ]:
pip install pandas faiss-cpu chromadb sentence-transformers fastapi uvicorn requests

In [1]:
import pandas as pd

# Load CSV file
df = pd.read_csv("error_codes_cleaned.csv")

# Combine columns into a single content field for retrieval
df["content"] = df.apply(lambda x: f"error_code: {x['error_code']}\ndescription: {x['description']}\nremedy: {x['remedy']}", axis=1)

# Drop NaN values
df = df.dropna().reset_index(drop=True)
df.to_csv("final_preprocessed.csv")
# Display sample
df.head()


PermissionError: [Errno 13] Permission denied: 'final_preprocessed.csv'

In [3]:
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np

# Load a lightweight embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert text content into embeddings
embeddings = model.encode(df["content"].tolist(), convert_to_numpy=True)

# Create a FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# Save FAISS index
faiss.write_index(index, "error_codes_faiss.index")

# Save CSV for later reference
df.to_csv("error_codes_with_embeddings.csv", index=False)


c:\Users\Lenovo\anaconda3\envs\Python10001\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def retrieve_similar(query, top_k=3):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)
    
    results = []
    for i in range(top_k):
        if distances[0][i] < 0.5:  # Adjust similarity threshold
            results.append(df.iloc[indices[0][i]]["content"])
    
    return results


In [5]:
import requests

GROQ_API_KEY = "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"
GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"

def generate_response(query):
    retrieved_info = retrieve_similar(query)
    
    if not retrieved_info:
        return "Sorry, I couldn't find relevant error codes. Please provide more details."

    context = "\n\n".join(retrieved_info)
    
    prompt = f"""You are a helpful Siemens PLC troubleshooting assistant. Based on the following Siemens error code details, provide guidance:\n\n{context}\n\nUser Query: {query}\n\nResponse:"""
    
    headers = {"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"}

    payload = {
        "model": "llama3-8b-8192",
        "messages": [
            {"role": "system", "content": "You are a Siemens PLC error code assistant."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.5
    }

    response = requests.post(GROQ_API_URL, headers=headers, json=payload)

    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"]
    else:
        return f"Error: {response.json()}"
